# ray-parametric-form composite — cx5: guard against rays parallel to plane via identity replacement

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ray-parametric-form`, `singular-matrix-mask-trick`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "ray-parametric-form"
DD_ATOM_IDS = ["ray-parametric-form", "singular-matrix-mask-trick"]
DD_SUBTOPICS = ["Geometry: Ray parametric form", "Numpy: Singular matrix mask trick"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A ray `P(t) = O + t*D` (the parametric form) misses a plane entirely when `D` is parallel to the plane — `n · D == 0`. In a batched ray-plane intersection that's the SAME failure mode as a singular coefficient matrix: the 1x1 system `[n·D] * t = n·(Q-O)` has determinant zero.

The `singular-matrix-mask-trick` works identically at the 1x1 scale: detect singular slices, replace them with the identity (here just `1.0`), let the solve succeed, then expose validity via a boolean mask so the caller never trusts the garbage `t` value for parallel rays.

Composition flow:
1. Compute `nd = n · D` per ray (the load-bearing scalar of `ray-parametric-form` at the plane).
2. Build a `(NR, 1, 1)` matrix; detect `|nd| < eps`.
3. Overwrite singular slots with `eye(1)`, solve, then mask out garbage solutions.
4. Evaluate `P = O + t*D` again — but only the masked-valid rows are real intersections.

### Composite Exercise — guard against rays parallel to plane via identity replacement

**Atoms exercised together**: `ray-parametric-form`, `singular-matrix-mask-trick`

Implement `cx5_safe_ray_plane(rays, n_plane, q_plane, eps=1e-8)` — a ray-plane intersection that gracefully handles rays parallel to the plane.

Inputs:
- `rays`: `(NR, 2, 3)` — `rays[r, 0]` is origin, `rays[r, 1]` is direction.
- `n_plane`: `(3,)` — plane normal.
- `q_plane`: `(3,)` — point on plane.
- `eps`: detection threshold on `|n · D|`.

Returns `(t_vals, points, hit)`:
- `t_vals: (NR,)` — solved parameters (garbage where `hit=False`).
- `points: (NR, 3)` — `O + t*D` (garbage where `hit=False`, but always FINITE).
- `hit: (NR,) bool` — `True` where the ray actually intersects (i.e. `|n·D| >= eps`).

**Algorithm:**
1. Compute `nd[r] = n_plane · D_r`, shape `(NR,)`.
2. `is_parallel = nd.abs() < eps`.
3. Build coefficient `A: (NR, 1, 1)` from `nd`; CLONE then overwrite singular slots with `1.0`.
4. Build RHS `b: (NR, 1)` from `n_plane · (Q - O)`.
5. `t_solved = solve(A_safe, b).squeeze(-1)` → `(NR,)`.
6. `points = O + t_solved.unsqueeze(-1) * D` (the parametric-form atom again).
7. Return `t_solved, points, ~is_parallel`.

In [ ]:
def cx5_safe_ray_plane(rays, n_plane, q_plane, eps=1e-8):
    O = rays[:, 0]              # (NR, 3)
    D = rays[:, 1]              # (NR, 3)
    NR = O.shape[0]
    # n . D per ray — zero iff D is parallel to the plane.
    nd = (D * n_plane).sum(dim=-1)                  # (NR,)
    is_parallel = nd.abs() < eps
    # Build coefficient (NR, 1, 1) and apply singular-matrix-mask-trick.
    A = nd.reshape(NR, 1, 1).clone()
    A[is_parallel] = t.eye(1)                       # broadcast (1,1) into masked slots
    # RHS.
    rhs = ((q_plane - O) * n_plane).sum(dim=-1).reshape(NR, 1)
    # linalg.solve — safe now.
    t_solved = t.linalg.solve(A, rhs).squeeze(-1)   # (NR,)
    # ray-parametric-form: P = O + t * D.
    points = O + t_solved.unsqueeze(-1) * D
    return t_solved, points, ~is_parallel


<details><summary>Show solution — cx5</summary>

```python
def cx5_safe_ray_plane(rays, n_plane, q_plane, eps=1e-8):
    O = rays[:, 0]              # (NR, 3)
    D = rays[:, 1]              # (NR, 3)
    NR = O.shape[0]
    # n . D per ray — zero iff D is parallel to the plane.
    nd = (D * n_plane).sum(dim=-1)                  # (NR,)
    is_parallel = nd.abs() < eps
    # Build coefficient (NR, 1, 1) and apply singular-matrix-mask-trick.
    A = nd.reshape(NR, 1, 1).clone()
    A[is_parallel] = t.eye(1)                       # broadcast (1,1) into masked slots
    # RHS.
    rhs = ((q_plane - O) * n_plane).sum(dim=-1).reshape(NR, 1)
    # linalg.solve — safe now.
    t_solved = t.linalg.solve(A, rhs).squeeze(-1)   # (NR,)
    # ray-parametric-form: P = O + t * D.
    points = O + t_solved.unsqueeze(-1) * D
    return t_solved, points, ~is_parallel
```

Parallel rays are the geometric face of singular coefficient matrices in ray-plane intersection. The mask trick at the 1x1 scale just replaces `n·D ≈ 0` with `1.0` — the solve succeeds and produces the harmless 'solution' `t = n·(Q-O)`, which downstream consumers IGNORE because `hit=False`.

Why this is the right armor: a naive `t.linalg.solve` on the original `(NR, 1, 1)` batch would either crash (PyTorch raises LinAlgError on the whole batch as soon as one slice is singular) or, if you switched to per-ray division `t = rhs / nd`, you'd get `inf` or `nan` for parallel rays and corrupt every downstream broadcast that touches them. The mask + identity-replace keeps EVERYTHING finite, and the validity flag is exposed to the caller.

Note that `ray-parametric-form` appears twice in the solution: implicitly in `n · D` (where `D` is the parametric direction vector), and explicitly in `P = O + t*D` at the end.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx5',
        'subtopics': ["Geometry: Ray parametric form", "Numpy: Singular matrix mask trick"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()